In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# ===== 입력 파일들 (업로드된 경로) =====
files = [
    "../data/f_clean_clean_hyperblick.csv",
    "../data/f_clean_clean_koreamasa.csv",
    "../data/f_clean_gmnara.csv",
    "../data/f_clean_makangs.csv",
    "../data/f_clean_naviya.csv",
    "../data/f_clean_roomhubs_seoul.csv",
]

# ===== 기준 스키마(컬럼 순서 고정) =====
SCHEMA = [
    "name","category","address_std","gu","sigu","lat","lng","is_valid_location",
    "COL_ADM_SE","station","station_exit","station_detail"
]

def read_csv_safely(fp: str) -> pd.DataFrame:
    # UTF-8 -> CP949 -> EUC-KR 순서로 시도
    for enc in ("utf-8-sig", "utf-8", "cp949", "euc-kr"):
        try:
            return pd.read_csv(fp, encoding=enc)
        except Exception:
            pass
    # 그래도 실패하면 에러를 올려 원인 확인
    return pd.read_csv(fp)

dfs = []
for fp in files:
    df = read_csv_safely(fp)

    # 혹시 스키마가 살짝 달라도 맞춰서 결합되도록 보정
    for c in SCHEMA:
        if c not in df.columns:
            df[c] = pd.NA
    df = df[SCHEMA]

    df["source_file"] = Path(fp).name

    dfs.append(df)

merged = pd.concat(dfs, ignore_index=True)

# out_path = "../data/crwaling_merged.csv"
out_path = "../data/test_crwaling_merged.csv"    # 시연용
merged.to_csv(out_path, index=False, encoding="utf-8-sig")

merged.shape, out_path

((825, 13), '../data/test_crwaling_merged.csv')

In [3]:
#  통합 데이터 개요 및 미리보기
df = pd.read_csv("../data/test_crwaling_merged.csv")

# 1. 전체 행 개수
print("총 행 개수:", len(df))

# 2. 위경도 확보 현황
if "lat" in df.columns:
    print("\n위경도 확보 건수:", df["lat"].notna().sum())
    print("위경도 결측 건수:", df["lat"].isna().sum())

# 3. is_valid_location 비율: True - 실제 주소 / False - 추정 위치
print("실제/추정 각각 개수: ", df["is_valid_location"].value_counts(dropna=False))
print("실제/추정 비율: ", df["is_valid_location"].value_counts(normalize=True, dropna=False))


# 4. name + address 기준 단순 중복 확인
if {"name", "address_std"}.issubset(df.columns):
    dup_count = df.duplicated(subset=["name", "address_std"]).sum()
    print("\nname + address 기준 중복 개수:", dup_count)

# 5. 상위 5개 행 미리보기
print("\n데이터 미리보기:")
display(df.sample())

총 행 개수: 825

위경도 확보 건수: 793
위경도 결측 건수: 32
실제/추정 각각 개수:  is_valid_location
True     596
False    229
Name: count, dtype: int64
실제/추정 비율:  is_valid_location
True     0.722424
False    0.277576
Name: proportion, dtype: float64

name + address 기준 중복 개수: 6

데이터 미리보기:


,name,category,address_std,gu,sigu,lat,lng,is_valid_location,COL_ADM_SE,station,station_exit,station_detail,source_file
677,세이렌,스웨디시 | 테라피,서울특별시 강남구 역삼동 670-18 테헤란로27길 22,강남구,서울특별시 강남구,37.50291,127.037734,True,11230.0,NaN,NaN,서울특별시 강남구 역삼동 670-18 (테헤란로27길 22),f_clean_naviya.csv
